# FortiOS 8 KVM Lab - Colab Client
## Connect to Remote Lab Service from Google Colab

This notebook connects to a running FortiOS 8 KVM lab service (hosted on Docker, Fly.io, or Render) and provides interactive control.

**Prerequisites:**
1. Lab service running somewhere (Docker locally, Fly.io, Render, etc.)
2. Service URL and API key

---

## STEP 1: Configure Service Connection

In [ ]:
# Service Configuration
SERVICE_URL = "http://localhost:5000"     # Change to your service URL
API_KEY = ""                              # Enter your API key
TARGET_IP = "127.0.0.1"                  # Target IP (localhost if service local)
TARGET_PORT = 12443                       # Tunnel port
TARGET_DEVICE = "FG-LAB-000001"          # Device serial

# Examples for different deployment methods:
# Local Docker:  http://localhost:5000
# Fly.io:        https://your-app.fly.dev
# Render:        https://your-app.onrender.com
# Custom:        https://your-domain.com:5000

print(f"✓ Configuration loaded")
print(f"  Service: {SERVICE_URL}")
print(f"  Target:  {TARGET_IP}:{TARGET_PORT}")
print(f"  Device:  {TARGET_DEVICE}")

## STEP 2: Install Dependencies

In [ ]:
import requests
import json
import time
from datetime import datetime

print("✓ All imports successful")

# Suppress SSL warnings for self-signed certs
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

## STEP 3: Lab Control Functions

In [ ]:
class LabClient:
    """Client for KVM lab service"""

    def __init__(self, url, api_key):
        self.url = url.rstrip('/')
        self.headers = {
            'X-API-Key': api_key,
            'Content-Type': 'application/json'
        }
        self.session = requests.Session()
        self.session.verify = False

    def _call(self, method, endpoint, data=None):
        """Make API call"""
        try:
    url = f"{self.url}{endpoint}"
            resp = self.session.request(method, url, json=data, headers=self.headers, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            return {'error': str(e)}

    def health(self):
        return self._call('GET', '/health')

    def connect(self):
        return self._call('POST', '/api/lab/connect')

    def setup(self):
        return self._call('POST', '/api/lab/setup')

    def tunnel_start(self):
        return self._call('POST', '/api/lab/tunnel/start')

    def tunnel_stop(self):
        return self._call('POST', '/api/lab/tunnel/stop')

    def exploit(self, target_ip, target_port, target_device):
        return self._call('POST', '/api/exploit', {
            'target_ip': target_ip,
            'target_port': target_port,
            'target_device': target_device
        })

    def recon(self, target_ip, target_port):
        return self._call('POST', '/api/exploit/recon', {
            'target_ip': target_ip,
            'target_port': target_port
        })

    def cleanup(self):
        return self._call('POST', '/api/lab/cleanup')

    def status(self):
        return self._call('GET', '/api/state')

# Create client
client = LabClient(SERVICE_URL, API_KEY)
print("✓ Lab client initialized")

## STEP 4: Check Service Connection

In [ ]:
print(f"🔍 Testing connection to {SERVICE_URL}...\n")

result = client.health()

if 'error' in result:
    print(f"❌ Service unavailable: {result['error']}")
    print(f"\n💡 Make sure the service is running:")
    print(f"   Local: python3 kvm_lab_server.py")
    print(f"   Docker: docker-compose up -d")
    print(f"   Fly.io: fly deploy")
    print(f"   Render: Connect GitHub repo")
else:
    print("✅ Service connection successful!")
    state = result.get('state', {})
    print(f"\n📊 Lab Status:")
    print(f"   KVM Connected:   {'✓' if state.get('kvm_connected') else '✗'}")
    print(f"   Tunnel Running:  {'✓' if state.get('tunnel_running') else '✗'}")
    print(f"   Exploit Success: {'✓' if state.get('exploit_results', {}).get('success') else '✗'}")

## STEP 5: Lab Operations

In [ ]:
# 1. Connect to KVM
print("[1/6] Connecting to KVM host...")
result = client.connect()
if result.get('success'):
    print(f"✓ Connected via {result.get('method')}\n")
else:
    print(f"✗ Failed: {result.get('error')}\n")

In [ ]:
# 2. Setup FortiOS 8
print("[2/6] Setting up FortiOS 8...")
result = client.setup()
if result.get('success'):
    print("✓ Setup complete\n")
else:
    print(f"✗ Failed: {result.get('error')}\n")

In [ ]:
# 3. Start SSH Tunnel
print("[3/6] Starting SSH tunnel...")
result = client.tunnel_start()
if result.get('success'):
    print(f"✓ {result.get('message')}\n")
else:
    print(f"✗ Failed: {result.get('error')}\n")

In [ ]:
# 4. Run Exploitation
print(f"[4/6] Running exploitation against {TARGET_IP}:{TARGET_PORT}...")
result = client.exploit(TARGET_IP, TARGET_PORT, TARGET_DEVICE)

if result.get('success'):
    print(f"✅ Exploitation successful!")
    print(f"   Vector: {result.get('vector')}")
    print(f"   Cookie: {result.get('cookie', '')[:50]}...\n")
else:
    print("❌ Exploitation failed")
    for r in result.get('results', []):
        status = '✓' if r.get('success') else '✗'
        print(f"   {status} {r.get('vector')}: {r.get('status_code')}")
    print()

In [ ]:
# 5. Post-Exploitation Recon
print(f"[5/6] Running reconnaissance...")
result = client.recon(TARGET_IP, TARGET_PORT)

if result.get('success'):
    recon = result.get('recon', {})
    print("✅ Reconnaissance complete\n")

    # System info
    if recon.get('system_info'):
        info = recon['system_info']
        print("📊 System Information:")
        print(f"   Version:  {info.get('version')}")
        print(f"   Serial:   {info.get('serial')}")
        print(f"   Hostname: {info.get('hostname')}")
        print()

    # Admin users
    if recon.get('admin_users'):
        users = recon['admin_users']
        print(f"👤 Admin Users ({len(users)}):")
        for user in users:
            print(f"   - {user.get('name', 'N/A'):20} {user.get('accprofile', 'N/A')}")
        print()

    # Firewall policies
    if recon.get('firewall_policies'):
        policies = recon['firewall_policies']
        print(f"🔥 Firewall Policies ({len(policies)}):")
        for policy in policies[:5]:
            print(f"   - {policy.get('policyid')}: {policy.get('name')}")
        if len(policies) > 5:
            print(f"   ... and {len(policies) - 5} more")
        print()
else:
    print(f"❌ Recon failed: {result.get('error')}\n")

In [ ]:
# 6. Cleanup
print("[6/6] Cleaning up...")
result = client.cleanup()
if result.get('success'):
    print("✓ Cleanup complete\n")
    print("🎉 Lab workflow finished!")
else:
    print(f"✗ Cleanup failed: {result.get('error')}\n")

## STEP 6: View Lab Status

In [ ]:
result = client.status()
if 'error' not in result:
    state = result.get('state', {})
    print("📋 Current Lab Status:")
    print(json.dumps(state, indent=2))
else:
    print(f"Error: {result.get('error')}")

## Manual Control (Use as Needed)

Use these cells individually to control the lab manually:


In [ ]:
# Manual: Check health
result = client.health()
print(json.dumps(result, indent=2))

In [ ]:
# Manual: Stop tunnel only
result = client.tunnel_stop()
print(json.dumps(result, indent=2))

In [ ]:
# Manual: Full cleanup
result = client.cleanup()
print(json.dumps(result, indent=2))

## Notes

- **Service must be running** before using this notebook
- Update `SERVICE_URL` and `API_KEY` in STEP 1
- Works with local Docker, Fly.io, Render, or custom deployments
- Each cell can be run independently for manual control
- Check cell outputs for status and errors

---

**Created:** July 23, 2026  
**Status:** Lab-Tested  
**Security:** Authorized Testing Only